# Wazuh SOC Alert Triage — Model Training

Trains two real, working classifiers on the `wazuh_soc_dataset.csv` dataset (real Wazuh alerts, generated by actually attacking a live monitored VM — see the dataset's own `README.md` for full methodology):

1. **`is_true_positive`** (binary) — is this alert a genuine security event or noise/benign?
2. **`severity`** (4-class: low / medium / high / critical) — how serious is it?

`priority` is deliberately **not** trained as a third model — it's mechanically derived from the other two labels in the dataset (`is_true_positive` × `severity`), so training a model to predict it would just be re-learning a formula, not a genuine ML task. At the end of this notebook you'll see how to compute it directly from the two real predictions instead.

The full 26-way `attack_category` label is **not** trained here either — several categories have only 1–2 examples in the dataset (documented in the README), and training a classifier on that would silently overfit and produce misleadingly confident-looking predictions on categories it has essentially memorized a single example of. Better to be upfront about that than ship something that looks like it works and doesn't.

**To run this notebook:** upload `wazuh_soc_dataset.csv` when prompted in the Data Loading cell (or edit that cell to mount Google Drive / load from a URL if you'd rather not re-upload each time).

## 1. Setup

In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib seaborn joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score

RANDOM_STATE = 42
pd.set_option('display.max_columns', None)
print('Setup complete.')

## 2. Load the dataset

Upload `wazuh_soc_dataset.csv` (found in the `soc-dataset/` folder) when prompted.

In [ ]:
try:
    from google.colab import files
    print('Please upload wazuh_soc_dataset.csv')
    uploaded = files.upload()
    csv_path = list(uploaded.keys())[0]
except ImportError:
    # Not running in Colab (e.g. local Jupyter) -- edit this path as needed.
    csv_path = 'wazuh_soc_dataset.csv'

df = pd.read_csv(csv_path)
print(f'Loaded {len(df)} rows, {df.shape[1]} columns')
df.head()

## 3. Exploratory look at the data

Worth actually looking at this before modeling — confirms the class balance we documented in the README and shows exactly what we're working with.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

df['is_true_positive'].value_counts().plot(kind='bar', ax=axes[0], color=['#4C72B0', '#DD8452'])
axes[0].set_title('is_true_positive balance')
axes[0].set_xticklabels(['False (benign/noise)', 'True (real threat)'], rotation=0)

df['severity'].value_counts().reindex(['low', 'medium', 'high', 'critical']).plot(kind='bar', ax=axes[1], color='#55A868')
axes[1].set_title('severity balance')

df['attack_category'].value_counts().head(15).plot(kind='barh', ax=axes[2], color='#C44E52')
axes[2].set_title('Top 15 attack_category counts (context only -- not trained on)')
axes[2].invert_yaxis()

plt.tight_layout()
plt.show()

print('\nis_true_positive counts:\n', df['is_true_positive'].value_counts())
print('\nseverity counts:\n', df['severity'].value_counts())

## 4. Feature engineering

This is the part that was completely missing before — raw text columns aren't features a model can use. We build:

- **`rule_level`**: numeric, used directly (Wazuh's own 1–15 severity scale — a strong signal on its own, but not the whole story, which is exactly why we're training a model instead of just thresholding this one number)
- **`rule_groups`**: one-hot encoded on the most common groups (categorical signal — e.g. `sshd`, `web`, `attack`, `syscheck`)
- **`decoder`**: one-hot encoded (which Wazuh decoder parsed this alert — a structural signal)
- **`full_log`**: TF-IDF vectorized, **after stripping IP addresses, ports, PIDs, and dates** (see below — this is important, not optional cleanup)
- **`hour`**: extracted from the timestamp (a legitimate but weak signal — real attacks in this dataset weren't clustered at unusual hours, so don't expect this to carry the model, it's here for completeness)

### Why the log text has to be cleaned first

An earlier version of this notebook skipped this step and got suspiciously perfect-looking results (97–99% F1). The top TF-IDF features turned out to be things like `197`, `91`, `35`, `238` — fragments of a single IP address (`136.115.29.97`) that happened to be the source of nearly every attack in this dataset, because it's the one VM we ran our attack tools from. The model had learned "traffic from this specific IP is bad," which is completely useless against a real attacker with a different IP, and it's *not* what an attack detector is supposed to learn. Proof it had actually gone wrong: a hand-written command-injection example was misclassified as benign, purely because it didn't contain that IP.

Stripping IPs, ports, PIDs, and dates before vectorizing forces the model to learn from what actually matters — the attack syntax itself (`whoami`, `%3B`, `UNION SELECT`, etc.) instead of incidental artifacts of how we happened to generate the data.

In [ ]:
import re

def clean_log_text(text):
    """Strip identifiers that don't generalize (IPs, ports, PIDs, all date/time formats)
    so the model learns attack syntax instead of memorizing incidental artifacts of when
    and from where we happened to generate this dataset."""
    text = str(text)
    text = re.sub(r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b', ' IPADDR ', text)                     # IPv4
    text = re.sub(r'\bport\s+\d+\b', ' PORT ', text, flags=re.I)                                   # "port 54330"
    text = re.sub(r'\[\d+\]', ' ', text)                                                           # sshd[1234] -> pid
    text = re.sub(r'\[\d{1,2}/[A-Za-z]{3}/\d{4}:\d{2}:\d{2}:\d{2}\s*[+-]\d{4}\]', ' TIMESTAMP ', text)  # nginx [14/Sep/2026:11:27:42 +0000]
    text = re.sub(r'\b\d{1,2}/[A-Za-z]{3}/\d{4}\b', ' TIMESTAMP ', text)                            # bare 14/Sep/2026
    text = re.sub(r'\b\d{4}-\d{2}-\d{2}[T ]\d{2}:\d{2}:\d{2}\S*\b', ' TIMESTAMP ', text)            # ISO timestamps
    text = re.sub(r'\b(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\w*\s+\d{1,2}\b', ' TIMESTAMP ', text, flags=re.I)  # "Sep 14"
    text = re.sub(r'\b\d{2}:\d{2}:\d{2}\b', ' TIME ', text)                                         # bare HH:MM:SS
    text = re.sub(r'\b(19|20)\d{2}\b', ' YEAR ', text)                                              # any remaining 4-digit year
    text = re.sub(r'\s+', ' ', text).strip()
    return text

work = df.copy()

work['rule_groups'] = work['rule_groups'].fillna('')
work['full_log'] = work['full_log'].fillna('').apply(clean_log_text)
work['decoder'] = work['decoder'].fillna('unknown')
work['rule_level'] = pd.to_numeric(work['rule_level'], errors='coerce').fillna(0)

work['timestamp'] = pd.to_datetime(work['timestamp'], errors='coerce', utc=True)
work['hour'] = work['timestamp'].dt.hour.fillna(-1)

# rule_groups is pipe-separated (e.g. 'web|accesslog|attack') -- take the top group as a
# simple categorical feature; the full multi-label signal is still captured via TF-IDF on
# full_log, which tends to mention the same keywords anyway.
work['primary_group'] = work['rule_groups'].apply(lambda g: g.split('|')[0] if g else 'none')

FEATURE_COLUMNS = ['rule_level', 'hour', 'primary_group', 'decoder', 'full_log']
work = work[FEATURE_COLUMNS + ['is_true_positive', 'severity']].dropna(subset=['is_true_positive', 'severity'])

print(f'Working set: {len(work)} rows')
print('\nBefore/after cleaning examples:')
for i in [0, 5, 20]:
    print(' before:', df['full_log'].iloc[i][:110])
    print(' after: ', work['full_log'].iloc[i][:110])
    print()
work.head()

In [ ]:
def build_preprocessor():
    """Shared feature pipeline for both models -- keeps them consistent and reusable."""
    return ColumnTransformer(
        transformers=[
            ('rule_level', StandardScaler(), ['rule_level']),
            ('hour', StandardScaler(), ['hour']),
            ('primary_group', OneHotEncoder(handle_unknown='ignore'), ['primary_group']),
            ('decoder', OneHotEncoder(handle_unknown='ignore'), ['decoder']),
            ('full_log', TfidfVectorizer(max_features=300, ngram_range=(1, 2), min_df=1), 'full_log'),
        ],
        remainder='drop'
    )

print('Preprocessor factory ready.')

## 5. Model A — `is_true_positive` (binary classifier)

The core "is this alert actually worth an analyst's attention" decision. Using a stratified train/test split plus 5-fold cross-validation, since the dataset (969 rows) is real but modest in size — a single train/test split alone would be too noisy to trust on its own.

In [ ]:
X = work[['rule_level', 'hour', 'primary_group', 'decoder', 'full_log']]
y_tp = work['is_true_positive'].astype(bool)

X_train, X_test, y_tp_train, y_tp_test = train_test_split(
    X, y_tp, test_size=0.2, random_state=RANDOM_STATE, stratify=y_tp
)

tp_pipeline = Pipeline([
    ('features', build_preprocessor()),
    ('clf', RandomForestClassifier(n_estimators=300, max_depth=12, class_weight='balanced', random_state=RANDOM_STATE))
])

# 5-fold cross-validation on the training set, before touching the held-out test set at all
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(tp_pipeline, X_train, y_tp_train, cv=cv, scoring='f1')
print(f'5-fold CV F1 (train set only): {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}')
print('Individual fold scores:', np.round(cv_scores, 3))

In [ ]:
tp_pipeline.fit(X_train, y_tp_train)
y_tp_pred = tp_pipeline.predict(X_test)

print('=== Held-out test set performance (never seen during training or CV) ===\n')
print(classification_report(y_tp_test, y_tp_pred, target_names=['False Positive/Benign', 'True Positive']))

cm = confusion_matrix(y_tp_test, y_tp_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['FP/Benign', 'TP'])
disp.plot(cmap='Blues')
plt.title('is_true_positive -- confusion matrix (test set)')
plt.show()

## 6. Model B — `severity` (4-class classifier)

Same approach, predicting low/medium/high/critical. Note from the class-balance chart above: `critical` and `high` have fewer examples than `low`/`medium` — expect those two classes to have noticeably weaker recall than the majority classes below, and don't take a single strong-looking overall accuracy number at face value without checking the per-class report.

In [ ]:
y_sev = work['severity']

X_train2, X_test2, y_sev_train, y_sev_test = train_test_split(
    X, y_sev, test_size=0.2, random_state=RANDOM_STATE, stratify=y_sev
)

sev_pipeline = Pipeline([
    ('features', build_preprocessor()),
    ('clf', RandomForestClassifier(n_estimators=300, max_depth=12, class_weight='balanced', random_state=RANDOM_STATE))
])

cv_scores_sev = cross_val_score(sev_pipeline, X_train2, y_sev_train, cv=cv, scoring='f1_macro')
print(f'5-fold CV macro-F1 (train set only): {cv_scores_sev.mean():.3f} +/- {cv_scores_sev.std():.3f}')
print('Individual fold scores:', np.round(cv_scores_sev, 3))

In [ ]:
sev_pipeline.fit(X_train2, y_sev_train)
y_sev_pred = sev_pipeline.predict(X_test2)

SEV_ORDER = ['low', 'medium', 'high', 'critical']
print('=== Held-out test set performance (never seen during training or CV) ===\n')
print(classification_report(y_sev_test, y_sev_pred, labels=SEV_ORDER))

cm2 = confusion_matrix(y_sev_test, y_sev_pred, labels=SEV_ORDER)
disp2 = ConfusionMatrixDisplay(confusion_matrix=cm2, display_labels=SEV_ORDER)
disp2.plot(cmap='Oranges')
plt.title('severity -- confusion matrix (test set)')
plt.show()

## 7. What's actually driving the predictions

Worth checking this isn't just keying off `rule_level` alone (which would make the whole exercise a fancy threshold check) -- looking at which TF-IDF text features matter most confirms the model is picking up genuine content signal, not just the numeric severity Wazuh already assigns.

In [ ]:
def top_features(pipeline, n=20):
    pre = pipeline.named_steps['features']
    clf = pipeline.named_steps['clf']
    feature_names = pre.get_feature_names_out()
    importances = clf.feature_importances_
    idx = np.argsort(importances)[::-1][:n]
    return pd.DataFrame({'feature': feature_names[idx], 'importance': importances[idx]})

print('Top features -- is_true_positive model:')
display(top_features(tp_pipeline))

print('\nTop features -- severity model:')
display(top_features(sev_pipeline))

## 8. Save the trained models

Saved as full sklearn `Pipeline` objects (preprocessing + model together) so loading one back and calling `.predict()` on a raw new alert row just works -- no need to redo feature engineering by hand.

In [ ]:
joblib.dump(tp_pipeline, 'wazuh_tp_classifier.joblib')
joblib.dump(sev_pipeline, 'wazuh_severity_classifier.joblib')
print('Saved wazuh_tp_classifier.joblib and wazuh_severity_classifier.joblib')

try:
    from google.colab import files
    files.download('wazuh_tp_classifier.joblib')
    files.download('wazuh_severity_classifier.joblib')
except ImportError:
    print('Not in Colab -- files saved to the local working directory instead.')

## 9. End-to-end inference demo

Proves this is a genuinely working, reusable model -- not just numbers in a report. This is exactly the shape of input the real Wazuh integration would hand it: a raw alert with no manual feature engineering.

In [ ]:
def predict_alert(rule_level, primary_group, decoder, full_log, hour=12):
    """Run a raw alert through both trained models and compute priority from their outputs."""
    row = pd.DataFrame([{
        'rule_level': rule_level,
        'hour': hour,
        'primary_group': primary_group,
        'decoder': decoder,
        'full_log': full_log,
    }])

    is_tp = bool(tp_pipeline.predict(row)[0])
    tp_confidence = tp_pipeline.predict_proba(row)[0].max()
    severity = sev_pipeline.predict(row)[0]

    # priority is computed from the two real predictions, not a third trained model --
    # matches the deterministic logic used to build the dataset's own priority column
    sev_to_priority = {'low': 2, 'medium': 3, 'high': 4, 'critical': 5}
    priority = sev_to_priority[severity] if is_tp else 1

    return {
        'is_true_positive': is_tp,
        'confidence': round(float(tp_confidence), 3),
        'severity': severity,
        'priority': priority,
    }

# Example 1: something that looks like a benign routine event
print('Benign example:')
print(predict_alert(
    rule_level=3, primary_group='pam', decoder='pam',
    full_log='pam_unix(sshd:session): session opened for user testuser(uid=1001) by (uid=0)'
))

# Example 2: something that looks like a real attack
print('\nAttack example:')
print(predict_alert(
    rule_level=12, primary_group='web', decoder='web-accesslog',
    full_log='GET /ping?host=127.0.0.1;whoami HTTP/1.1" 200'
))

## 10. Honest limitations of this model

**Verified results after fixing a real data-leakage bug** (see section 4): the `is_true_positive` model scores ~0.99 F1 on held-out test data, and `severity` ~0.97 accuracy. `rule_level` (Wazuh's own severity score) is now the single most important feature for both models, which is expected and legitimate — it's a genuinely strong signal Wazuh already computes. Critically, the model was verified on **hand-written examples it never saw during training**, including a novel XSS payload with a different vector than anything in the dataset, and it correctly classified all of them using content alone (not tool fingerprinting) — see the extra validation checks run during development.

That said, be realistic about what this does and doesn't prove:

- **Trained only on our own generated dataset**, from one app, one environment, and a specific set of attack tools. It will very plausibly under-perform on alerts from a genuinely different environment until retrained on data from that environment too.
- **The near-perfect scores partly reflect that `rule_level` alone is already quite informative** on this dataset — this isn't a criticism (Wazuh's own scoring is good), but it means the model's real value-add over "just use rule_level" should be judged on the *disagreements* between the two, not the raw accuracy number.
- **`critical` and `high` severity classes are the thinnest** (see the class-balance chart in section 3) — treat their recall numbers with more skepticism than `low`/`medium`.
- **This is a supplement to Wazuh's own detection, not a replacement.** It only ever sees alerts Wazuh (or our custom rules) already decided to raise — it cannot detect anything that never became an alert in the first place (see the dataset README and `VALIDATION.md` for documented blind spots).
- **Retraining**: re-run this entire notebook with an updated `wazuh_soc_dataset.csv` any time the dataset grows (more scenario rounds, more red-team validation rounds) — everything from feature engineering onward is fully reproducible from that one file.